In [5]:
import torch
import torch.nn as nn
import random
import numpy as np

# Simple 4x4 grid world: agent starts at (0,0) and goal is at (3,3)
class GridWorld:
    def __init__(self):
        self.size = 4
        self.reset()
    
    def reset(self):
        self.pos = [0, 0]
        return self.get_state()
    
    def get_state(self):
        return self.pos[0] * self.size + self.pos[1]
    
    def step(self, action):
        # Actions: 0=up, 1=right, 2=down, 3=left
        if action == 0 and self.pos[0] > 0:
            self.pos[0] -= 1
        elif action == 1 and self.pos[1] < self.size - 1:
            self.pos[1] += 1
        elif action == 2 and self.pos[0] < self.size - 1:
            self.pos[0] += 1
        elif action == 3 and self.pos[1] > 0:
            self.pos[1] -= 1
        
        done = (self.pos == [3, 3])
        reward = 10 if done else -0.1
        return self.get_state(), reward, done

# Simple Q-Network
class QNetwork(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_size, 24),
            nn.ReLU(),
            nn.Linear(24, 24),
            nn.ReLU(),
            nn.Linear(24, action_size)
        )
    
    def forward(self, x):
        return self.fc(x)

# Training
env = GridWorld()
state_size = 16  # 4x4 grid
action_size = 4
q_net = QNetwork(state_size, action_size)
optimizer = torch.optim.Adam(q_net.parameters(), lr=0.001)
gamma = 0.95
epsilon = 1.0

for episode in range(500):
    state = env.reset()
    total_reward = 0
    
    for step in range(50):
        # Epsilon-greedy action selection
        if random.random() < epsilon:
            action = random.randint(0, 3)
        else:
            with torch.no_grad():
                state_tensor = torch.zeros(state_size)
                state_tensor[state] = 1
                action = q_net(state_tensor).argmax().item()
        
        next_state, reward, done = env.step(action)
        total_reward += reward
        
        # Q-learning update
        state_tensor = torch.zeros(state_size)
        state_tensor[state] = 1
        next_state_tensor = torch.zeros(state_size)
        next_state_tensor[next_state] = 1
        
        with torch.no_grad():
            target = reward + gamma * q_net(next_state_tensor).max()
        
        q_values = q_net(state_tensor)
        loss = (q_values[action] - target) ** 2
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        state = next_state
        if done:
            break
    
    epsilon = max(0.01, epsilon * 0.995)
    
    if episode % 100 == 0:
        print(f"Episode {episode}, Total Reward: {total_reward:.2f}, Epsilon: {epsilon:.3f}")

print("\nTraining complete!")

Episode 0, Total Reward: -5.00, Epsilon: 0.995
Episode 100, Total Reward: 8.90, Epsilon: 0.603
Episode 200, Total Reward: 9.50, Epsilon: 0.365
Episode 300, Total Reward: 9.40, Epsilon: 0.221
Episode 400, Total Reward: 9.50, Epsilon: 0.134

Training complete!


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import numpy as np

# Simple 4x4 grid world: agent starts at (0,0) and goal is at (3,3)
class GridWorld:
    def __init__(self):
        self.size = 4
        self.reset()
    
    def reset(self):
        self.pos = [0, 0]
        return self.get_state()
    
    def get_state(self):
        return self.pos[0] * self.size + self.pos[1]
    
    def step(self, action):
        # Actions: 0=up, 1=right, 2=down, 3=left
        if action == 0 and self.pos[0] > 0:
            self.pos[0] -= 1
        elif action == 1 and self.pos[1] < self.size - 1:
            self.pos[1] += 1
        elif action == 2 and self.pos[0] < self.size - 1:
            self.pos[0] += 1
        elif action == 3 and self.pos[1] > 0:
            self.pos[1] -= 1
        
        done = (self.pos == [3, 3])
        reward = 10 if done else -0.1
        return self.get_state(), reward, done

# Policy Network - outputs probability distribution over actions
class PolicyNetwork(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_size, 24),
            nn.ReLU(),
            nn.Linear(24, 24),
            nn.ReLU(),
            nn.Linear(24, action_size)
        )
    
    def forward(self, x):
        return F.softmax(self.fc(x), dim=-1)

# Training with REINFORCE (policy gradient)
env = GridWorld()
state_size = 16
action_size = 4
policy_net = PolicyNetwork(state_size, action_size)
optimizer = torch.optim.Adam(policy_net.parameters(), lr=0.01)
gamma = 0.99

for episode in range(1000):
    state = env.reset()
    log_probs = []
    rewards = []
    
    # Collect episode trajectory
    for step in range(50):
        state_tensor = torch.zeros(state_size)
        state_tensor[state] = 1
        
        # Sample action from policy (probability distribution)
        action_probs = policy_net(state_tensor)
        action_dist = torch.distributions.Categorical(action_probs)
        action = action_dist.sample()
        
        log_probs.append(action_dist.log_prob(action))
        
        next_state, reward, done = env.step(action.item())
        rewards.append(reward)
        state = next_state
        
        if done:
            break
    
    # Calculate returns (discounted cumulative rewards)
    returns = []
    G = 0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    returns = torch.tensor(returns)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    
    # Policy gradient update
    policy_loss = []
    for log_prob, G in zip(log_probs, returns):
        policy_loss.append(-log_prob * G)
    
    optimizer.zero_grad()
    loss = torch.stack(policy_loss).sum()
    loss.backward()
    optimizer.step()
    
    if episode % 200 == 0:
        total_reward = sum(rewards)
        print(f"Episode {episode}, Total Reward: {total_reward:.2f}")

print("\nPolicy learning complete!")

Episode 0, Total Reward: 7.20
Episode 200, Total Reward: 9.50
Episode 400, Total Reward: -5.00
Episode 600, Total Reward: -5.00
Episode 800, Total Reward: -5.00

Policy learning complete!
